<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/marco-canas/linea_invest_didact_math_data/blob/main/1_estrategia_comunicacion/3_multiconferencia/1_propuesta_multiconferencia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
  </td>
  <td>
    <a target="_blank" href="https://kaggle.com/kernels/welcome?src=https://github.com/marco-canas/linea_invest_didact_math_data/blob/main/1_estrategia_comunicacion/3_multiconferencia/1_propuesta_multiconferencia.ipynb"><img src="https://kaggle.com/static/images/open-in-kaggle.svg" /></a>
  </td>
</table>

# **Propuesta de Trabajo Reflexivo para Humanos XXI** 

 
**Título:**  
*"Investigación-Acción Educativa (IAE) en la Enseñanza del Álgebra Lineal para Ingeniería Agropecuaria: Un Enfoque hacia la Ciencia de Datos y la Optimización Agrícola"*  


In [ ]:
from docx import Document
from docx.shared import Pt, Inches, RGBColor
from docx.oxml.ns import qn
from docx.enum.text import WD_ALIGN_PARAGRAPH, WD_PARAGRAPH_ALIGNMENT
from docx.enum.style import WD_STYLE_TYPE
import re

def apply_apa_reference_style(paragraph):
    """Aplica estilo APA a referencias bibliográficas"""
    paragraph.paragraph_format.left_indent = Inches(0.5)
    paragraph.paragraph_format.first_line_indent = Inches(-0.5)
    paragraph.style = 'Normal'
    if paragraph.runs:  # Verificar que hay runs
        run = paragraph.runs[0]
        run.font.name = 'Times New Roman'
        run._element.rPr.rFonts.set(qn('w:eastAsia'), 'Times New Roman')
        run.font.size = Pt(11)
        run.font.color.rgb = RGBColor(0, 0, 0)

def add_numbered_heading(doc, text, level):
    """Añade encabezados numerados con estilo formal"""
    # Configurar contadores
    if level == 1:
        if not hasattr(doc, '_section_counter'):
            doc._section_counter = 0
        doc._section_counter += 1
        doc._subsection_counter = 0
        number = f"{doc._section_counter}."
    elif level == 2:
        if not hasattr(doc, '_subsection_counter'):
            doc._subsection_counter = 0
        doc._subsection_counter += 1
        doc._subsubsection_counter = 0
        number = f"{doc._section_counter}.{doc._subsection_counter}"
    elif level == 3:
        if not hasattr(doc, '_subsubsection_counter'):
            doc._subsubsection_counter = 0
        doc._subsubsection_counter += 1
        number = f"{doc._section_counter}.{doc._subsection_counter}.{doc._subsubsection_counter}"
    else:
        number = ""
    
    # Crear encabezado con estilo
    heading = doc.add_heading('', level=level)
    run = heading.add_run(f"{number} {text}")
    run.font.name = 'Times New Roman'
    run._element.rPr.rFonts.set(qn('w:eastAsia'), 'Times New Roman')
    run.font.size = Pt(14 if level == 1 else 12)
    run.font.bold = True
    run.font.color.rgb = RGBColor(0, 0, 0)
    
    # Alineación según normas del congreso
    if level == 1:
        heading.alignment = WD_ALIGN_PARAGRAPH.CENTER
    else:
        heading.alignment = WD_ALIGN_PARAGRAPH.LEFT
    
    return heading

def process_markdown_text(text):
    """Convierte formato markdown a estilos Word"""
    # Negritas
    text = re.sub(r'\*\*(.*?)\*\*', r'\1', text)
    text = re.sub(r'__(.*?)__', r'\1', text)
    # Cursivas
    text = re.sub(r'\*(.*?)\*', r'\1', text)
    text = re.sub(r'_(.*?)_', r'\1', text)
    # Links
    text = re.sub(r'\[(.*?)\]\((.*?)\)', r'\1 (\2)', text)
    # Listas
    text = re.sub(r'^\s*-\s', '• ', text, flags=re.MULTILINE)
    text = re.sub(r'^\s*\*\s', '• ', text, flags=re.MULTILINE)
    # Encabezados markdown
    text = re.sub(r'^#+\s*(.*)', r'\1', text, flags=re.MULTILINE)
    return text

def add_section(doc, title_text, body_lines, level=1):
    """Añade una sección con contenido formateado"""
    add_numbered_heading(doc, title_text, level)
    
    for line in body_lines:
        line = process_markdown_text(line.strip())
        if not line:
            doc.add_paragraph()  # Párrafo vacío
            continue
            
        p = None  # Inicializar variable p
        
        if line.startswith('•') or (line[0].isdigit() and line[1] == '.'):
            # Lista con viñetas o numerada
            list_style = 'List Bullet' if line.startswith('•') else 'List Number'
            p = doc.add_paragraph(style=list_style)
            p.add_run(line[2:].strip())
        elif ':' in line and len(line.split(':', 1)[0]) < 30:
            # Término en negrita seguido de definición
            term, definition = line.split(':', 1)
            p = doc.add_paragraph()
            p.add_run(term.strip() + ':').bold = True
            p.add_run(definition.strip())
        else:
            # Párrafo normal
            p = doc.add_paragraph(line)
        
        # Aplicar estilos si p fue creado
        if p:
            for run in p.runs:
                run.font.name = 'Times New Roman'
                run._element.rPr.rFonts.set(qn('w:eastAsia'), 'Times New Roman')
                run.font.size = Pt(12)

def create_document_from_markdown(md_content):
    """Crea documento Word desde contenido markdown"""
    doc = Document()
    
    # Configurar estilos base
    style = doc.styles['Normal']
    style.font.name = 'Times New Roman'
    style._element.rPr.rFonts.set(qn('w:eastAsia'), 'Times New Roman')
    style.font.size = Pt(12)
    
    # Extraer secciones del markdown
    sections = re.split(r'^##+\s', md_content, flags=re.MULTILINE)[1:]
    
    # Procesar cada sección
    for section in sections:
        if not section.strip():
            continue
            
        # Separar título y contenido
        title_end = section.find('\n')
        title = section[:title_end].strip()
        content = section[title_end:].strip()
        
        # Determinar nivel de encabezado
        level = 1  # Por defecto nivel 1
        
        # Procesar contenido
        body_lines = [line for line in content.split('\n')]
        
        # Añadir sección al documento
        add_section(doc, title, body_lines, level)
    
    return doc


Documento generado exitosamente: propuesta_congreso_humanos_xxi.docx


In [ ]:

def main():
    try:
        # Leer el archivo markdown
        with open('propuesta_en_20_paginas.md', 'r', encoding='utf-8') as f:
            md_content = f.read()
        
        # Crear documento Word
        doc = create_document_from_markdown(md_content)
        
        # Añadir portada según normas del congreso
        doc.add_section()
        title = doc.add_heading('', level=0)
        title_run = title.add_run('"Investigación-Acción Educativa (IAE) en la Enseñanza del Álgebra Lineal para Ingeniería Agropecuaria: Un Enfoque hacia la Ciencia de Datos y la Optimización Agrícola"')
        title_run.bold = True
        title_run.font.size = Pt(16)
        title_run.font.name = 'Times New Roman'
        title.alignment = WD_ALIGN_PARAGRAPH.CENTER
        
        # Autoría
        doc.add_paragraph().alignment = WD_ALIGN_PARAGRAPH.CENTER
        author = doc.add_paragraph()
        author.alignment = WD_ALIGN_PARAGRAPH.CENTER
        author.add_run('Marco Julio Cañas Campillo\n').bold = True
        author.add_run('Universidad de Antioquia, Dirección de Regionalización\n')
        author.add_run('Profesor Facultad de Ciencias Agrarias, Campus Caucasia\n')
        author.add_run('Docente Ocasional de Tiempo Completo\n')
        author.add_run('Investigador — Grupo GIBACC\n')
        
        # Congreso
        congreso = doc.add_paragraph()
        congreso.alignment = WD_ALIGN_PARAGRAPH.CENTER
        congreso.add_run('Congreso Humanos XXI (7-9 de octubre, modalidad virtual)\n').italic = True
        congreso.add_run('https://fundacioniai.org/humanosxxi/').font.color.rgb = RGBColor(0, 0, 255)
        
        doc.add_page_break()
        
        # Guardar documento
        output_file = 'propuesta_congreso_humanos_xxi.docx'
        doc.save(output_file)
        print(f'Documento generado exitosamente: {output_file}')
    
    except FileNotFoundError:
        print("Error: No se encontró el archivo 'propuesta_en_20_paginas.md'")
    except Exception as e:
        print(f"Error inesperado: {str(e)}")

if __name__ == '__main__':
    main()

